# Nutrition5k: experimentos MobileNet

Notebook fino: solo llama a `python -m nutrition5k ...`. Toda la lógica está en `src/nutrition5k/`.

- Funciona en local, en Colab y en un pod de RunPod (abierto con VS Code Remote-SSH o con el JupyterLab del pod).
  En Colab, `outputs/` se enlaza a Google Drive; en RunPod, al volumen de red (`/workspace/nutrition5k`); en local se usa `code/outputs/`.
- Cada ejecución guarda `config.json` (hiperparámetros y entorno), `history.csv` (una fila por época: pérdidas, PMAE de validación,
  segundos y memoria de GPU), `best.pt`, `last.pt`, predicciones y métricas de validación y test.
- Volver a ejecutar una celda es seguro: los runs terminados se saltan y los interrumpidos se reanudan desde `last.pt`.
- Los datos se guardan en Drive o en el volumen como un único `.tar` y se descomprimen en el disco local, porque leer
  miles de JPEG desde Drive o desde un volumen de red es lento.
- Seguimiento en vivo: si la variable `WANDB_API_KEY` existe (en RunPod, como secreto del pod), cada run se refleja en
  wandb.ai (proyecto `nutrition5k`). Sin ella, mirar `history.csv` con `03_results.ipynb`.
- Las decisiones (qué backbone o configuración gana) se toman con **validación** en `03_results.ipynb`, nunca con test.

## Parámetros

In [ ]:
import os

FULL = True          # False: piloto 32/8 para pruebas rápidas
MODE = "complete"    # "complete" (cada 5º frame, todas las cámaras) o "minimal" (1 frame por plato)
EPOCHS = 30
WORKERS = min(8, os.cpu_count())  # procesos que decodifican JPEG; Colab suele tener 2 vCPU

FULL_FLAG = "--full" if FULL else ""
SPLIT = "full" if FULL else "pilot"

## Preparación del entorno

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    COLAB = True
except ImportError:
    COLAB = False
RUNPOD = "RUNPOD_POD_ID" in os.environ
REMOTE = COLAB or RUNPOD

REPOSITORY_URL = "https://github.com/victorabellotorres/thesis-multimodal-health-ai.git"
# Local disk of the session: fast, but lost when the session or pod ends.
REPOSITORY_DIR = Path("/content/thesis-multimodal-health-ai" if COLAB else "/root/thesis-multimodal-health-ai")
# Persistent storage: Google Drive on Colab, the network volume on RunPod.
PERSIST_ROOT = Path("/content/drive/MyDrive/nutrition5k" if COLAB else "/workspace/nutrition5k")

if REMOTE:
    if (REPOSITORY_DIR / ".git").is_dir():
        !git -C {REPOSITORY_DIR} pull --ff-only
    else:
        !git clone {REPOSITORY_URL} {REPOSITORY_DIR}
    %cd {REPOSITORY_DIR}
    %pip install -q -e .[train]
elif Path.cwd().name == "notebooks":
    # Local: the package is already installed in code/.venv; run from the repository root.
    os.chdir("..")

# The kernel's interpreter, so `!` commands use the same environment locally (VS Code may not activate .venv).
PYTHON = sys.executable

if COLAB:
    drive.mount("/content/drive")
if REMOTE:
    # Results live on Drive or the volume so nothing is lost when the session ends.
    (PERSIST_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
    if not Path("outputs").exists():
        Path("outputs").symlink_to(PERSIST_ROOT / "outputs")
print("W&B:", "on" if os.environ.get("WANDB_API_KEY") else "off (only local files)")

En local, `fetch` y `prepare` trabajan directamente sobre `data/` y se saltan lo que ya está descargado.

En Colab y RunPod, si el `.tar` ya existe en Drive o en el volumen se restaura; si no, `prepare` descarga los vídeos por lotes de 100 platos, extrae los frames reducidos a 320 px y borra los vídeos de cada lote. Después se guarda el `.tar` (solo frames) para la próxima sesión. En RunPod lo normal es subir el `.tar` preparado en el portátil (ver el README), así la GPU no espera a las descargas.

Si `prepare` falla (por ejemplo, un timeout que persiste tras los reintentos), no se crea el `.tar`. Si `prepare` falla o se interrumpe pero la sesión sigue viva, volver a ejecutar la celda continúa donde se quedó. Si la sesión se cierra antes de crear el `.tar`, el disco local se pierde y la preparación empieza de nuevo.

In [ ]:
DATA_TAR = PERSIST_ROOT / f"data-{SPLIT}-{MODE}.tar"

if REMOTE and DATA_TAR.is_file():
    !tar -xf {DATA_TAR}
else:
    !{PYTHON} -m nutrition5k fetch
    !{PYTHON} -m nutrition5k prepare --mode {MODE} {FULL_FLAG}
    if REMOTE and _exit_code == 0:  # a failed prepare must not be cached as a partial .tar
        !tar -cf {DATA_TAR} --exclude='*.h264' data  # frames only, not the videos
!{PYTHON} -m nutrition5k check --mode {MODE} {FULL_FLAG}

## Baseline (media del conjunto de entrenamiento)

In [ ]:
!{PYTHON} -m nutrition5k baseline {FULL_FLAG}

In [ ]:
def train(backbone, unfreeze="all", heads="single", hidden=(512,), image_size=224, seed=0, all_seeds=False):
    """Run one configuration; finished runs are skipped, interrupted ones resume."""
    hidden_args = " ".join(f"--hidden {size}" for size in (hidden or (0,)))
    command = (
        f"{PYTHON} -m nutrition5k train --backbone {backbone} --unfreeze {unfreeze} "
        f"--heads {heads} {hidden_args} --image-size {image_size} --epochs {EPOCHS} "
        f"--num-workers {WORKERS} --seed {seed} --mode {MODE} {FULL_FLAG}"
    )
    if all_seeds:
        command += " --all-seeds"
    print(command)
    get_ipython().system(command)

## Etapa 1: comparación de backbones

Configuración por defecto: fine-tuning completo, cabeza `single`, capa oculta compartida de 512, 224 px.

In [ ]:
for backbone in ("v2", "v3", "v4s", "v4m"):
    train(backbone)

## Etapa 2: ablaciones secuenciales sobre el mejor backbone

Tras cada paso, mirar `03_results.ipynb`, elegir el ganador **por validación** y escribirlo en la variable `BEST_*` del paso siguiente.
Los valores por defecto ya ejecutados en la etapa 1 se reutilizan, no se repiten.

### Paso 1: profundidad del fine-tuning (0 = extracción de características, N = últimas N etapas, all = completo)

In [ ]:
BEST_BACKBONE = "v4s"  # rellenar con el ganador de la etapa 1

for unfreeze in (0, 1, 2):
    train(BEST_BACKBONE, unfreeze=unfreeze)

### Paso 2: agrupación de cabezas (single / grouped = masa, calorías, macros / per-target)

In [ ]:
BEST_UNFREEZE = "all"  # rellenar con el ganador del paso 1

for heads in ("grouped", "per-target"):
    train(BEST_BACKBONE, BEST_UNFREEZE, heads=heads)

### Paso 3: tamaño del tronco compartido

In [ ]:
BEST_HEADS = "single"  # rellenar con el ganador del paso 2

for hidden in ((), (1024, 1024)):
    train(BEST_BACKBONE, BEST_UNFREEZE, BEST_HEADS, hidden=hidden)

### Paso 4: resolución de entrada

In [ ]:
BEST_HIDDEN = (512,)  # rellenar con el ganador del paso 3

train(BEST_BACKBONE, BEST_UNFREEZE, BEST_HEADS, BEST_HIDDEN, image_size=256)

## Etapa 3: semillas

Repetir la configuración final (y las que queden a menos del ruido entre semillas) con las semillas fijas 0, 1 y 2.
La semilla 0 ya está hecha y se reutiliza. `--all-seeds` escribe la media ± SD en `outputs/mobilenet/seeds/<configuración>.json`.

In [ ]:
BEST_IMAGE_SIZE = 224  # rellenar con el ganador del paso 4

train(BEST_BACKBONE, BEST_UNFREEZE, BEST_HEADS, BEST_HIDDEN, BEST_IMAGE_SIZE, all_seeds=True)